# 03 — NUMA placement and socket scaling

**Question.** What does it cost to use memory on the other socket, how well do two sockets scale when run independently, and what happens when a single process spans both?

Remote and local refer to the memory socket relative to the CPU socket. Two-socket results use two processes timed over one shared clock window (at least 99% overlap required) unless stated as a single spanning process.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('03_numa_and_socket_scaling')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say = ctx.table, ctx.figure, ctx.say

## 1. Local against remote memory

**Conditions.** Flat mode, 56 cores of one socket, 8 GiB, memory bound to the local or the other socket's tier. Read-only and non-temporal Triad show how the penalty depends on traffic direction.

In [ ]:
loc = report.locality_ratio(ext, thesis)
fig = plotting.go.Figure()
for cfg in ['flat_ddr', 'flat_hbm']:
    part = loc[loc.config == cfg].groupby('kernel')[['local', 'remote']].mean()
    for placement, dash in [('local', None), ('remote', 'dot')]:
        fig.add_bar(x=[f'{k}' for k in part.index], y=part[placement], name=f"{plotting.CONFIG_LABELS[cfg]} · {placement}",
                    marker_color=plotting.CONFIG_COLORS[cfg], opacity=1.0 if placement == 'local' else 0.45)
plotting.style(fig, 'Local and remote bandwidth · socket 0 and 1 averaged', 'Kernel', 'GB/s', 480)
fig.update_layout(barmode='group')
figure(fig, 'local_remote_bandwidth')
table(loc.groupby(['config', 'kernel'], as_index=False)[['local', 'remote', 'remote_over_local']].mean())

In [ ]:
g = loc.groupby(['config', 'kernel'])[['remote_over_local']].mean().remote_over_local
say(f"**Result.** Remote DDR retains {g[('flat_ddr','read')]:.0%} (read) to {g[('flat_ddr','triad_nt')]:.0%} (non-temporal Triad) of local DDR bandwidth. "
    f"Remote HBM retains only {g[('flat_hbm','read')]:.0%} to {g[('flat_hbm','triad_nt')]:.0%} of local HBM, which is {loc[(loc.config=='flat_hbm')&(loc.kernel=='read')].remote.mean():.0f} GB/s: below remote DDR.")

**Implication.** HBM is only an advantage when accessed from its own socket: the inter-socket link, not the HBM stacks, is the limit for remote accesses, and remote HBM reads are slower than remote DDR reads. Any placement that lets threads fault HBM pages on the wrong socket loses an order of magnitude. Always bind memory and CPUs to the same socket for bandwidth-bound work.

## 2. Two sockets, two processes

Each pair completes initialization and warm-up, then synchronizes readiness before a shared clock window. Efficiency is the pair's aggregate throughput divided by the sum of the isolated socket rates.

In [ ]:
de = report.dual_efficiency(ext, thesis)
de['label'] = de.benchmark.str.replace('_peak', '') + ' ' + de.dtype.fillna('') + ' ' + de.memory_mode
de['mean'] = de.config
view = de[de.working_set_gib.isna() | (de.working_set_gib <= 16)].copy()
fig = plotting.go.Figure()
for mode, part in view.groupby('memory_mode'):
    fig.add_bar(x=part.label.str.replace(' ' + mode, '', regex=False) + ' · ' + part.campaign_id.str.split('-').str[1], y=part.efficiency,
                name=plotting.CONFIG_LABELS['cache' if mode == 'cache' else 'flat_hbm'].split(' ·')[0],
                marker_color='#268270' if mode == 'cache' else '#C24A40')
fig.add_hline(y=1.0, line_dash='dot', line_color='#6B7480')
plotting.style(fig, 'Two-socket scaling efficiency · dual rate ÷ sum of isolated sockets', 'Kernel · campaign', 'Efficiency', 520)
fig.update_layout(barmode='group')
fig.update_yaxes(range=[0.9, 1.05])
figure(fig, 'dual_socket_efficiency')
table(de.drop(columns=['label', 'mean', 'metric']))

In [ ]:
say(f"**Result.** Across {len(de)} paired measurements the two-process rate is {de.efficiency.min():.3f}–{de.efficiency.max():.3f}× the sum of isolated sockets "
    f"(median {de.efficiency.median():.3f}); the shared window overlapped by at least {de.overlap_min.min():.1%}. Compute paths cluster at 0.985–0.992, bandwidth paths at or just above 1.")

**Implication.** Independent sockets do not interfere: a node delivers the sum of its sockets for compute (about 1% below, consistent with a slightly lower clock when all cores run) and bandwidth. This describes independent workers; it does not measure tensor-parallel scaling or cross-socket communication.

## 3. One process spanning both sockets

LLM servers that do not pin to a socket run one process with threads on all 112 cores. Two placements are compared for 16 GiB of data: *interleaved* pages across both sockets' memory, and *first-touch* (each thread allocates local to its socket).

In [ ]:
sp = report.span_table(ext)
table(sp[['memory_mode', 'config', 'benchmark', 'placement', 'k', 'gbs', 'gops', 'cv_percent', 'n_samples']])

In [ ]:
mem = sp[sp.benchmark.isin(['stream_read', 'stream_triad']) & sp.placement.isin(['interleave', 'firsttouch'])]
fig = plotting.go.Figure()
for placement, color in [('interleave', '#3476A6'), ('firsttouch', '#C24A40')]:
    part = mem[mem.placement == placement]
    fig.add_bar(x=part.config + ' · ' + part.benchmark.str.replace('stream_', ''), y=part.gbs, name=placement, marker_color=color)
dual_ref = report.bandwidth_summary(ext, thesis)
plotting.style(fig, 'Single process, 112 threads · bandwidth by page placement', 'Memory · kernel', 'GB/s', 500)
fig.update_layout(barmode='group')
figure(fig, 'span_bandwidth')

**Implication.** See the table: first-touch placement is what lets a single spanning process approach the two-process result, while interleaving halves the locality benefit by making half of every thread's traffic remote.

## 4. Cores needed to saturate one socket (October Triad)

In [ ]:
from hwbench.analysis import scaling_table, knees_table
core = thesis[(thesis.benchmark == 'stream_triad') & (thesis.notes == 'core_scaling') & (thesis.socket_scope == 'single') & (thesis.socket == 0) & (thesis.locality == 'local')]
c = core.groupby(['memory_mode', 'memory_target', 'threads']).bandwidth_gbs.median().reset_index()
c['config'] = [plotting.config_key(m, t) for m, t in zip(c.memory_mode, c.memory_target)]
fig = plotting.line_by_config(c, 'threads', 'bandwidth_gbs', 'config', 'Triad bandwidth against core count · socket 0, 8 GiB', 'Cores', 'GB/s')
figure(fig, 'triad_core_scaling')